## Intermediate steps of the tanh-gradient edge field (Method F's mechanism)

**Where it went**: this was the central machinery of Method F — a continuous edge field, NOT a binary boundary mask. The chain:

```
s_k(x)   = π_k(x) − max_{j≠k} π_j(x)        # signed margin per lineage
t_k(x)   = tanh(K_sharp · s_k)               # sharpened — saturates at ±1
e_k(x)   = ‖∇ t_k‖₂                          # Sobel gradient magnitude — peaks at sign-change
edge_total(x) = Σ_k e_k(x)                   # combined edge field over all lineages
weighted_edge(x) = edge_total · c² · evidence    # weighted by confidence and morphology
```

Then **threshold** the weighted_edge to get a binary boundary. This is conceptually different from argmax-flip boundary: it's a continuous field where the threshold knob trades sensitivity for off-target.

Below we show all the intermediates per lineage (focal pair only: Fib, Mel) plus the summed `edge_total` and `weighted_edge`.

In [ ]:
# Per-lineage tanh-gradient intermediates for the focal pair (Fib, Mel)
from scipy.ndimage import sobel
K_SHARP = lib.K_SHARP
top1_val = np.take_along_axis(pi_abst, top_idx[None].astype(np.int64), axis=0)[0]
masked   = np.where(np.arange(lib.K, dtype=np.int8)[:, None, None] == top_idx[None], -np.inf, pi_abst)
top2_val = np.max(masked, axis=0)

pair = [(ka, lin_a), (kb, lin_b)]
fig, axes = plt.subplots(len(pair), 4, figsize=(18, 4.5 * len(pair)))
for r, (k, L) in enumerate(pair):
    others_max = np.where(top_idx == k, top2_val, top1_val)
    s_k = pi_abst[k] - others_max          # signed margin
    t_k = np.tanh(K_SHARP * s_k).astype(np.float32)
    gy = sobel(t_k, axis=0); gx = sobel(t_k, axis=1)
    grad_t_k = np.hypot(gx, gy).astype(np.float32)
    sign_change = find_boundaries((s_k > 0).astype(np.uint8), mode="inner")

    vm_s = max(abs(s_k[zoom].min()), abs(s_k[zoom].max()))
    axes[r,0].imshow(s_k[zoom], cmap="RdBu_r", vmin=-vm_s, vmax=vm_s)
    axes[r,0].contour(find_boundaries(focal_z, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=1.2)
    axes[r,0].set_title(f"s_k = π[{L}] − max(others)\nmax in focal = {s_k[focal].max():.3f}", fontsize=10)
    axes[r,1].imshow(t_k[zoom], cmap="RdBu_r", vmin=-1, vmax=1)
    axes[r,1].contour(find_boundaries(focal_z, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=1.2)
    axes[r,1].set_title(f"tanh(K·s_k)  K={K_SHARP}", fontsize=10)
    axes[r,2].imshow(grad_t_k[zoom], cmap="hot", vmin=0, vmax=float(np.percentile(grad_t_k, 99.5)))
    axes[r,2].contour(find_boundaries(focal_z, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=1.2)
    axes[r,2].contour(sign_change[zoom].astype(int), levels=[0.5], colors="lime", linewidths=1.2)
    axes[r,2].set_title(f"|∇ tanh(K·s_k)|  (peaks at sign-change line in lime)", fontsize=10)
    # Overlay |∇ tanh| heatmap onto colored 18S to show input-on-context
    axes[r,3].imshow(colored_18s[zoom])
    ov = np.zeros((*grad_t_k[zoom].shape, 4), dtype=np.float32)
    g_n = grad_t_k[zoom] / max(grad_t_k[zoom].max(), 1e-9)
    ov[..., 0] = 1.0; ov[..., 3] = g_n * 0.85
    axes[r,3].imshow(ov, interpolation="none")
    axes[r,3].contour(find_boundaries(focal_z, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=1.2)
    axes[r,3].set_title(f"|∇ tanh| overlaid on colored 18S\n(red = gradient strength)", fontsize=10)
for ax in axes.ravel(): ax.set_xticks([]); ax.set_yticks([])
plt.suptitle(f"Tanh-gradient intermediates per lineage (focal pair {lin_a}, {lin_b}) — zoom on focal", fontsize=12, y=1.005)
plt.tight_layout(); plt.show()


## Gradient estimation breakdown — x / y components, magnitude, direction

What is `|∇ tanh(K · s_k)|` actually doing under the hood? Step through every piece:

```
gx_t = sobel(t_k, axis=1)         # x-derivative via 3×3 Sobel (includes small smoothing)
gy_t = sobel(t_k, axis=0)         # y-derivative
|∇t_k|  = sqrt(gx² + gy²)         # rotation-invariant gradient magnitude
∠∇t_k   = atan2(gy_t, gx_t)       # direction (which way the edge faces)
```

Two pairs to compare:
- **Gradient of the raw margin** `∇ s_k` — shows where the margin changes spatially before tanh
- **Gradient of the sharpened margin** `∇ tanh(K · s_k)` — the actual edge field; should look concentrated at the sign-change line because tanh has steepest slope there

Per-focal-lineage breakdown (Fib, Mel) at zoom on focal.

In [ ]:
from scipy.ndimage import sobel
from matplotlib.colors import hsv_to_rgb
K_SHARP = lib.K_SHARP

def gradient_pieces(field):
    """Return gx, gy, magnitude, angle (radians) for a 2D scalar field."""
    gx = sobel(field, axis=1).astype(np.float32)
    gy = sobel(field, axis=0).astype(np.float32)
    mag = np.hypot(gx, gy).astype(np.float32)
    ang = np.arctan2(gy, gx).astype(np.float32)
    return gx, gy, mag, ang

def angle_rgb(angle, mag, mag_max):
    """Encode gradient direction as hue, magnitude as value."""
    H = (angle + np.pi) / (2 * np.pi)
    S = np.ones_like(H)
    V = np.clip(mag / max(mag_max, 1e-9), 0, 1)
    hsv = np.stack([H, S, V], axis=-1)
    return hsv_to_rgb(hsv)

# Per-focal-lineage gradient breakdown
pair = [(ka, lin_a), (kb, lin_b)]
top1_val = np.take_along_axis(pi_abst, top_idx[None].astype(np.int64), axis=0)[0]
masked   = np.where(np.arange(lib.K, dtype=np.int8)[:, None, None] == top_idx[None], -np.inf, pi_abst)
top2_val = np.max(masked, axis=0)

fig, axes = plt.subplots(len(pair) * 2, 5, figsize=(22, 8 * len(pair)))
for ri, (k, L) in enumerate(pair):
    others_max = np.where(top_idx == k, top2_val, top1_val)
    s_k = pi_abst[k] - others_max
    t_k = np.tanh(K_SHARP * s_k).astype(np.float32)
    # Row A: gradient of RAW margin s_k
    gx_s, gy_s, mag_s, ang_s = gradient_pieces(s_k)
    # Row B: gradient of TANH-sharpened margin t_k
    gx_t, gy_t, mag_t, ang_t = gradient_pieces(t_k)
    sign_change = find_boundaries((s_k > 0).astype(np.uint8), mode="inner")

    vm_s = max(abs(s_k[zoom]).max(), 1e-6)
    for axA, (img, title, cmap, vmin, vmax) in zip(axes[2*ri, :], [
            (s_k,   f"s_k = π[{L}] − max(others)", "RdBu_r", -vm_s, vm_s),
            (gx_s,  f"∂s_k/∂x  (Sobel x)",          "RdBu_r", -mag_s.max(), mag_s.max()),
            (gy_s,  f"∂s_k/∂y  (Sobel y)",          "RdBu_r", -mag_s.max(), mag_s.max()),
            (mag_s, f"|∇ s_k|  (magnitude)",         "hot",   0,            float(np.percentile(mag_s, 99.5)) or 1),
            (angle_rgb(ang_s, mag_s, float(np.percentile(mag_s, 99.5)) or 1)[zoom],
                    f"∠∇ s_k  (hue=direction, value=magnitude)", None, None, None),
          ]):
        if cmap is None:
            axA.imshow(img)
        else:
            axA.imshow(img[zoom], cmap=cmap, vmin=vmin, vmax=vmax)
        axA.contour(find_boundaries(focal_z, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=1.2)
        axA.contour(sign_change[zoom].astype(int), levels=[0.5], colors="lime", linewidths=1.0)
        axA.set_title(title, fontsize=9)
        axA.set_xticks([]); axA.set_yticks([])

    for axB, (img, title, cmap, vmin, vmax) in zip(axes[2*ri + 1, :], [
            (t_k,   f"tanh(K · s_k)   K={K_SHARP}", "RdBu_r", -1, 1),
            (gx_t,  f"∂tanh/∂x  (Sobel x)",         "RdBu_r", -mag_t.max(), mag_t.max()),
            (gy_t,  f"∂tanh/∂y  (Sobel y)",         "RdBu_r", -mag_t.max(), mag_t.max()),
            (mag_t, f"|∇ tanh(K·s_k)|  ← the edge field per lineage", "hot", 0, float(np.percentile(mag_t, 99.5)) or 1),
            (angle_rgb(ang_t, mag_t, float(np.percentile(mag_t, 99.5)) or 1)[zoom],
                    f"∠∇ tanh  (hue=direction, value=magnitude)", None, None, None),
          ]):
        if cmap is None:
            axB.imshow(img)
        else:
            axB.imshow(img[zoom], cmap=cmap, vmin=vmin, vmax=vmax)
        axB.contour(find_boundaries(focal_z, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=1.2)
        axB.contour(sign_change[zoom].astype(int), levels=[0.5], colors="lime", linewidths=1.0)
        axB.set_title(title, fontsize=9)
        axB.set_xticks([]); axB.set_yticks([])

plt.suptitle("Gradient estimation pieces — per focal lineage. "
             "Rows: raw margin s_k then tanh(K·s_k). Cols: field / ∂x / ∂y / magnitude / direction.\n"
             "Yellow = focal contour; lime = sign-change line (where the gradient should peak).", fontsize=11, y=1.005)
plt.tight_layout(); plt.show()


### Gradient pieces for confidence² and cell_evidence (the multiplicative gates)

`weighted_edge = edge_total · c² · evidence`. The gradients OF c² and evidence matter too — they bound how sharply the edge field can rise or fall across the gating regions.

In [ ]:
c2 = (confidence ** 2).astype(np.float32)
for name, field in [("confidence²", c2), ("cell_evidence", evidence)]:
    gx, gy, mag, ang = gradient_pieces(field)
    fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
    axes[0].imshow(field[zoom], cmap="viridis", vmin=0, vmax=1)
    axes[0].set_title(f"{name} (zoom)", fontsize=10)
    axes[1].imshow(gx[zoom], cmap="RdBu_r", vmin=-mag.max(), vmax=mag.max())
    axes[1].set_title(f"∂{name}/∂x", fontsize=10)
    axes[2].imshow(gy[zoom], cmap="RdBu_r", vmin=-mag.max(), vmax=mag.max())
    axes[2].set_title(f"∂{name}/∂y", fontsize=10)
    axes[3].imshow(mag[zoom], cmap="hot", vmin=0, vmax=float(np.percentile(mag, 99.5)) or 1)
    axes[3].set_title(f"|∇ {name}|", fontsize=10)
    axes[4].imshow(angle_rgb(ang, mag, float(np.percentile(mag, 99.5)) or 1)[zoom])
    axes[4].set_title(f"∠∇ {name}", fontsize=10)
    for ax in axes:
        ax.contour(find_boundaries(focal_z, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=1.2)
        ax.set_xticks([]); ax.set_yticks([])
    plt.suptitle(f"Gradient pieces of {name} (one of the multiplicative gates on weighted_edge)", fontsize=11, y=1.04)
    plt.tight_layout(); plt.show()


In [ ]:
# Summed over all lineages: edge_total → multiplied by c² × evidence → weighted_edge
edge_total, _, _ = lib.edge_global_field(pi_abst)
weighted_edge = (edge_total * (confidence ** 2) * evidence).astype(np.float32)
print(f"edge_total: max={edge_total.max():.3f}, mean={edge_total.mean():.3f}")
print(f"weighted_edge: max={weighted_edge.max():.3f}, mean={weighted_edge.mean():.3f}, max in focal = {weighted_edge[focal].max():.3f}")

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
v_e  = float(np.percentile(edge_total, 99.5))
v_we = float(np.percentile(weighted_edge, 99.5))
# Full 2048 row
axes[0,0].imshow(edge_total, cmap="hot", vmin=0, vmax=v_e)
axes[0,0].contour(find_boundaries(focal, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=0.6)
axes[0,0].set_title(f"edge_total = Σ_k |∇ tanh(K·s_k)|\nmax = {edge_total.max():.2f}", fontsize=10)
axes[0,1].imshow(weighted_edge, cmap="hot", vmin=0, vmax=v_we)
axes[0,1].contour(find_boundaries(focal, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=0.6)
axes[0,1].set_title(f"weighted_edge = edge_total · c² · evidence\nmax = {weighted_edge.max():.2f}", fontsize=10)
axes[0,2].imshow(colored_18s)
ov = np.zeros((*weighted_edge.shape, 4), dtype=np.float32)
w_n = weighted_edge / max(weighted_edge.max(), 1e-9)
ov[..., 0] = 1.0; ov[..., 3] = w_n * 0.85
axes[0,2].imshow(ov, interpolation="none")
axes[0,2].contour(find_boundaries(focal, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=0.6)
axes[0,2].set_title("weighted_edge overlaid on colored 18S — full 2048", fontsize=10)
# Zoom row
axes[1,0].imshow(edge_total[zoom], cmap="hot", vmin=0, vmax=v_e)
axes[1,0].contour(find_boundaries(focal_z, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=1.3)
axes[1,0].set_title(f"edge_total — zoom\nmax in focal = {edge_total[focal].max():.2f}", fontsize=10)
axes[1,1].imshow(weighted_edge[zoom], cmap="hot", vmin=0, vmax=v_we)
axes[1,1].contour(find_boundaries(focal_z, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=1.3)
axes[1,1].set_title(f"weighted_edge — zoom\nmax in focal = {weighted_edge[focal].max():.2f}", fontsize=10)
axes[1,2].imshow(colored_18s[zoom])
ov_z = np.zeros((*weighted_edge[zoom].shape, 4), dtype=np.float32)
w_nz = weighted_edge[zoom] / max(weighted_edge.max(), 1e-9)
ov_z[..., 0] = 1.0; ov_z[..., 3] = w_nz * 0.85
axes[1,2].imshow(ov_z, interpolation="none")
axes[1,2].contour(find_boundaries(focal_z, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=1.3)
axes[1,2].set_title("weighted_edge on colored 18S — zoom", fontsize=10)
for ax in axes.ravel(): ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Tanh-gradient edge field — final summed quantities (input intensities adjacent)", fontsize=12, y=1.001)
plt.tight_layout(); plt.show()


## Method 7 — tanh-gradient boundary (threshold on weighted_edge)

Use the continuous `weighted_edge` field, threshold at a chosen percentile to declare boundary pixels. Knob: `τ_w` (percentile of weighted_edge ON the broader argmax boundary, or absolute threshold on the field). This is the natural "continuous" alternative to the discrete argmax-flip boundary.

In [ ]:
# Method 7: tanh-gradient threshold
def boundary_tanh_gradient(weighted_edge, percentile=99.0):
    """Boundary where weighted_edge exceeds its `percentile`-th value globally."""
    thr = float(np.percentile(weighted_edge, percentile))
    return weighted_edge > thr, thr

b7, thr7 = boundary_tanh_gradient(weighted_edge, percentile=99.0)
b7_p995, thr7_p995 = boundary_tanh_gradient(weighted_edge, percentile=99.5)
b7_p98,  thr7_p98  = boundary_tanh_gradient(weighted_edge, percentile=98.0)
print(f"Method 7 — boundary = weighted_edge > τ_w (percentile threshold)")
print(f"  τ_w @ p98:  {thr7_p98:.3f}  → {b7_p98.sum():,} px  ({b7_p98[focal].sum()} in focal)")
print(f"  τ_w @ p99:  {thr7:.3f}  → {b7.sum():,} px  ({b7[focal].sum()} in focal)")
print(f"  τ_w @ p99.5: {thr7_p995:.3f}  → {b7_p995.sum():,} px  ({b7_p995[focal].sum()} in focal)")


In [ ]:
# Visualize Method 7 at three thresholds — INPUT (weighted_edge) and OUTPUT (boundary) side-by-side
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
axes[0].imshow(weighted_edge[zoom], cmap="hot", vmin=0, vmax=v_we)
axes[0].contour(find_boundaries(focal_z, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=1.3)
axes[0].set_title(f"INPUT: weighted_edge (zoom)\nmax in focal = {weighted_edge[focal].max():.2f}", fontsize=10)
for ax, (b, thr, name) in zip(axes[1:], [(b7_p98, thr7_p98, "p98"), (b7, thr7, "p99"), (b7_p995, thr7_p995, "p99.5")]):
    ax.imshow(colored_18s[zoom])
    ov = np.zeros((*b[zoom].shape, 4), dtype=np.float32)
    ov[b[zoom]] = (0.0, 1.0, 1.0, 0.7)
    ax.imshow(ov, interpolation="none")
    ax.contour(find_boundaries(focal_z, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=1.3)
    ax.set_title(f"Method 7 boundary @ {name}\nτ_w = {thr:.3f}  n_focal = {b[focal].sum()}", fontsize=10)
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Method 7 — tanh-gradient threshold: input (weighted_edge) + 3 thresholds overlaid on colored 18S",
             fontsize=11, y=1.04)
plt.tight_layout(); plt.show()


# V0 sandbox — Layer 2 of 3: Boundary estimation (π_abst → boundary mask)

**Scope:** given the per-pixel lineage posterior from notebook 1, identify where the **heterotypic lineage boundaries** live.

**Failure mode the user flagged (2026-05-21):** the raw `find_boundaries(argmax(π))` mask fires anywhere the per-pixel argmax flips — including pixels with ~1 transcript locally where the lineage call is essentially random. Confidence²-along-boundary modulation reduces depth there but doesn't eliminate the boundary itself; we need to fix it at the boundary level.

**Candidate boundary detectors** tested in this notebook (all start from the same `π_abst`):
1. Raw `find_boundaries(argmax(π))` — what V0 currently does
2. Confidence-gated argmax boundary: `find_boundaries(argmax) & (confidence > τ_c)`
3. Margin-based: boundary = `margin < τ_m` (low-margin pixels)
4. Uncertainty-based: boundary = `(1 − π_top1) > τ_u`
5. Entropy-based: boundary = `H(π) > τ_h`

Plus combinations (e.g., argmax & cell_evidence > 0.1 to also exclude empty stroma).

**No cut, no CP-SAM here.** Output: a boundary mask we'll feed to notebook 3.

Test ROI: same DB_top100_018, 2048×2048 px.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
REPO = Path.cwd().parents[0]
sys.path.insert(0, str(REPO / "pipelines" / "V0"))
import numpy as np, pandas as pd, tifffile, matplotlib.pyplot as plt, zarr
from matplotlib.colors import ListedColormap
from scipy.ndimage import gaussian_filter
from skimage.segmentation import find_boundaries
import lib

## Setup — reproduce π_abst from notebook 1 (self-contained for iteration)

In [ ]:
BENCHMARK_ID = "DB_top100_031"  # brighter 18S; was 018 — top-50% median 1275 vs 566; CROP_PX = 384  # focused ROI; NB 1/2 don't run CP-SAM so we don't need the full 2048 context; ZOOM_PAD = 30  # tighter zoom inside the focused ROI
bench = pd.read_parquet(lib.DATA / "benchmark_doublets.parquet")
row = bench[bench.benchmark_id == BENCHMARK_ID].iloc[0]
cx, cy = float(row.x_um), float(row.y_um); cps_focal = int(row.cps_id_at_bookmark)
lin_a, lin_b = row.lineage_pair.split(" × ")
ka, kb = lib.LIN_TO_IDX[lin_a], lib.LIN_TO_IDX[lin_b]
with tifffile.TiffFile(lib.DAPI_TIF) as tf: wsi_H, wsi_W = tf.series[0].shape[-2:]
y0, y1, x0, x1 = lib.make_roi_bbox_centred(cx, cy, CROP_PX, wsi_H, wsi_W)
dapi, s18 = lib.load_morphology(y0, y1, x0, x1)
py, px, li = lib.load_anchors_in_bbox(y0, y1, x0, x1, lib.load_lineage_label_map())
H, W = s18.shape

# PRODUCTION posterior — anisotropic diffusion guided by 18S (σ=2 µm equivalent)
pi_abst, confidence, N_eff = lib.lineage_posterior_anisotropic(
    py, px, li, H, W, s18,
    alpha=lib.ALPHA, n_min=lib.N_MIN,
    n_iter=lib.N_ITER_ANISO, dt=lib.DT_ANISO,
)
top_idx = np.argmax(pi_abst, axis=0).astype(np.int8)
top1_val = np.take_along_axis(pi_abst, top_idx[None].astype(np.int64), axis=0)[0]
masked = np.where(np.arange(lib.K, dtype=np.int8)[:, None, None] == top_idx[None], -np.inf, pi_abst)
top2_val = np.max(masked, axis=0)
margin = (top1_val - top2_val).astype(np.float32)
evidence = lib.cell_evidence(dapi, s18, lib.load_wsi_percentiles())

wsi_mask_full = tifffile.imread(lib.DATA / "cpsam_whole_slide" / "masks.tif")
focal = (wsi_mask_full[y0:y1, x0:x1] == cps_focal); del wsi_mask_full
ys, xs = np.where(focal)
zy0 = max(int(ys.min())-ZOOM_PAD, 0); zy1 = min(int(ys.max())+ZOOM_PAD, H)
zx0 = max(int(xs.min())-ZOOM_PAD, 0); zx1 = min(int(xs.max())+ZOOM_PAD, W)
zoom = (slice(zy0, zy1), slice(zx0, zx1))
focal_z = focal[zoom]
vmax_s18 = float(np.percentile(s18, 99))
print(f"setup done: pi_abst {pi_abst.shape}, confidence mean {confidence.mean():.3f}, focal {focal.sum()} px ({lin_a}×{lin_b})")

## Direct input — Colored 18S

Every boundary detector below operates on the per-pixel lineage assignment from `π_abst`. The cleanest way to inspect that input is the **colored 18S** tissue map: 18S brightness × argmax-lineage colour, falling back to grayscale where confidence is low. This is what should drive any boundary call — and what we'll overlay boundary masks on, below.

In [ ]:
# Colored 18S — BLENDED weights (pi_ml = N_eff / N_total), local p5-p95 brightness, grayscale fallback
LIN_COLORS = lib.V0_LINEAGE_COLORS
LIN_COLOR_ARRAY = np.array(LIN_COLORS[:lib.K], dtype=np.float32)

def make_colored_18s(s18, N_eff, confidence, color_array, low_pct=5, high_pct=95):
    lo, hi = np.percentile(s18, [low_pct, high_pct])
    s18_n = np.clip((s18.astype(np.float32) - lo) / max(hi - lo, 1e-6), 0, 1)
    N_total = N_eff.sum(axis=0)
    pi_ml = N_eff / np.maximum(N_total, 1e-9)[None]
    color = np.einsum("khw,kc->hwc", pi_ml, color_array)
    tinted = color * s18_n[..., None] * confidence[..., None]
    gray   = np.stack([s18_n, s18_n, s18_n], axis=-1) * (1.0 - confidence[..., None])
    return (tinted + gray).astype(np.float32)

colored_18s = make_colored_18s(s18, N_eff, confidence, LIN_COLOR_ARRAY)
print(f"colored_18s ready: {colored_18s.shape}, range=[{colored_18s.min():.3f}, {colored_18s.max():.3f}]")


In [ ]:
# Colored 18S — the direct input to boundary detection (full 2048 + zoom)
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
axes[0].imshow(colored_18s)
axes[0].contour(find_boundaries(focal, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=0.8)
axes[0].set_title("COLORED 18S — full 2048\nThis is what every boundary detector below is operating on")
for k, L in enumerate(lib.LINEAGES):
    axes[0].scatter([], [], s=80, color=LIN_COLORS[k], label=L, edgecolor="black", linewidth=0.5)
axes[0].legend(loc="upper right", fontsize=9, framealpha=0.9)
axes[1].imshow(colored_18s[zoom])
axes[1].contour(find_boundaries(focal_z, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=1.5)
axes[1].set_title("COLORED 18S — zoom on focal\nyellow = focal cell footprint")
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


## Candidate boundary detectors

All return a boolean mask `(H, W)`. Each detector has its own knob(s) that we can tune.

In [ ]:
# 1. Raw argmax boundary (V0 baseline)
def boundary_argmax(top_idx):
    return find_boundaries(top_idx, mode="thick").astype(bool)

# 2. Confidence-gated argmax boundary
def boundary_argmax_conf(top_idx, confidence, tau_c=0.5):
    raw = find_boundaries(top_idx, mode="thick").astype(bool)
    return raw & (confidence > tau_c)

# 3. Evidence-gated argmax boundary (alternative: morphology gate)
def boundary_argmax_ev(top_idx, evidence, tau_e=0.10):
    raw = find_boundaries(top_idx, mode="thick").astype(bool)
    return raw & (evidence > tau_e)

# 4. Margin-based: low-margin pixels are heterotypic-zone candidates
#    (multiplied by confidence to avoid the empty-stroma flicker)
def boundary_margin(margin, confidence, tau_m=0.10, tau_c=0.5):
    return (margin < tau_m) & (confidence > tau_c)

# 5. Uncertainty-based
def boundary_uncertainty(top1_val, confidence, tau_u=0.6, tau_c=0.5):
    return ((1.0 - top1_val) > tau_u) & (confidence > tau_c)

# 6. Entropy-based
def boundary_entropy(pi_abst, confidence, tau_h=1.5, tau_c=0.5):
    eps = 1e-9
    H_pi = -(pi_abst * np.log(pi_abst + eps)).sum(axis=0)
    return (H_pi > tau_h) & (confidence > tau_c)

# Compute all candidates with reasonable defaults
boundaries = {
    "1. argmax (raw, V0 baseline)":       boundary_argmax(top_idx),
    "2. argmax & conf>0.5":                boundary_argmax_conf(top_idx, confidence, 0.5),
    "3. argmax & evidence>0.10":           boundary_argmax_ev(top_idx, evidence, 0.10),
    "4. margin<0.10 & conf>0.5":           boundary_margin(margin, confidence, 0.10, 0.5),
    "5. uncertainty (1-top1)>0.6 & conf>0.5":   boundary_uncertainty(top1_val, confidence, 0.6, 0.5),
    "6. entropy>1.5 & conf>0.5":           boundary_entropy(pi_abst, confidence, 1.5, 0.5),
}
for name, b in boundaries.items():
    print(f"  {name:48s}  n_px = {b.sum():>8,}  ({b.mean()*100:5.2f}% of frame)  inside focal = {b[focal].sum()}")

In [ ]:
# Visualize each boundary on 18S — full 2048
n = len(boundaries); ncols = 3; nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(7.5 * ncols, 7.5 * nrows))
axes = axes.ravel() if hasattr(axes, "ravel") else [axes]
for ax, (name, b) in zip(axes, boundaries.items()):
    ax.imshow(colored_18s)
    ov = np.zeros((*b.shape, 4), dtype=np.float32)
    ov[b] = (0.0, 1.0, 1.0, 0.75)  # cyan
    ax.imshow(ov, interpolation="none")
    ax.contour(find_boundaries(focal, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=0.8)
    ax.set_title(f"{name}\n{b.sum():,} px ({b.mean()*100:.2f}%)", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes[len(boundaries):]: ax.set_visible(False)
plt.suptitle("Boundary detectors — full 2048 (cyan = boundary; yellow = focal cell)", fontsize=12, y=1.005)
plt.tight_layout(); plt.show()

In [ ]:
# Same plot, ZOOM on focal cell, with anchors overlaid to see if boundary lands on dense-anchor regions
LIN_COLORS = lib.V0_LINEAGE_COLORS
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes = axes.ravel() if hasattr(axes, "ravel") else [axes]
in_z = (py >= zy0) & (py < zy1) & (px >= zx0) & (px < zx1)
for ax, (name, b) in zip(axes, boundaries.items()):
    ax.imshow(colored_18s[zoom])
    for k, L in enumerate(lib.LINEAGES):
        m = in_z & (li == k)
        if m.any():
            ax.scatter(px[m] - zx0, py[m] - zy0, s=18, color=LIN_COLORS[k],
                       edgecolors="black", linewidths=0.3, alpha=0.85)
    ov = np.zeros((*focal_z.shape, 4), dtype=np.float32)
    ov[b[zoom]] = (0.0, 1.0, 1.0, 0.6)
    ax.imshow(ov, interpolation="none")
    ax.contour(find_boundaries(focal_z, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=1.5)
    ax.set_title(f"{name}\n{b[focal].sum()} px inside focal", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes[len(boundaries):]: ax.set_visible(False)
plt.suptitle("Boundary detectors — zoom on focal cell with labeled anchors", fontsize=12, y=1.005)
plt.tight_layout(); plt.show()

## Threshold sweep — confidence gate τ_c

For the argmax & conf > τ_c family, see how much of the boundary survives at different τ_c values.

In [ ]:
taus = [0.0, 0.3, 0.5, 0.7, 0.9]
raw_b = find_boundaries(top_idx, mode="thick").astype(bool)
print(f"raw argmax boundary: {raw_b.sum():,} px ({raw_b.mean()*100:.2f}% of frame)")
print(f"{'τ_c':5s} | n_px        | in focal | mean N_total at boundary")
for tau in taus:
    b = raw_b & (confidence > tau)
    print(f"{tau:5.2f} | {b.sum():>11,} | {b[focal].sum():>8d} | {confidence[b].mean() if b.any() else 0:.3f}")

In [ ]:
# Visualize boundary at each τ_c — zoom on focal with anchors
fig, axes = plt.subplots(1, len(taus), figsize=(4.5 * len(taus), 5))
for ax, tau in zip(axes, taus):
    b = raw_b & (confidence > tau)
    ax.imshow(colored_18s[zoom])
    for k, L in enumerate(lib.LINEAGES):
        m = in_z & (li == k)
        if m.any():
            ax.scatter(px[m] - zx0, py[m] - zy0, s=14, color=LIN_COLORS[k],
                       edgecolors="black", linewidths=0.3, alpha=0.85)
    ov = np.zeros((*focal_z.shape, 4), dtype=np.float32)
    ov[b[zoom]] = (0.0, 1.0, 1.0, 0.6)
    ax.imshow(ov, interpolation="none")
    ax.contour(find_boundaries(focal_z, mode="outer").astype(int), levels=[0.5], colors="yellow", linewidths=1.5)
    ax.set_title(f"argmax & conf > {tau}\nn_focal_px = {b[focal].sum()}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Confidence-threshold sweep on argmax boundary (zoom on focal)", fontsize=12, y=1.04)
plt.tight_layout(); plt.show()

## Multi-doublet validation (more zoom-in examples)

Loop through several dev doublets covering the brightness range AND known failure modes. For each: re-load morphology + anchors, recompute the anisotropic posterior, then show the same per-doublet panel as above. Catches whether a given boundary detector behaves robustly across cells.

**Doublets**:
- DB_top100_031 — production test case (Fib × Mel, bright)
- DB_top100_026 — bright Fib × Mel
- DB_top100_018 — dimmer baseline (Fib × Mel)
- DB_top100_015 — known failure: dim 18S along the boundary itself (Endo × Fib)
- DB_top100_090 — known failure: anchor-derived boundary in wrong location (Mel × Myeloid)
- DB_top100_082 — very bright Fib × Plasma (different pair)

**Columns per doublet** (zoom on focal cell):
1. Colored 18S + anchors (INPUT)
2. argmax raw — Method 1 (V0 baseline)
3. argmax & conf > 0.5 — Method 2 (confidence-gated)
4. tanh-gradient @ p99 — Method 7

Anchors overlaid on every panel — the canonical visual check that the boundary lands at dense-anchor regions, not in low-confidence stroma.

In [ ]:
import time
from scipy.ndimage import sobel
from skimage.segmentation import find_boundaries

EXTRA_DOUBLETS = ["DB_top100_031", "DB_top100_026", "DB_top100_018",
                  "DB_top100_015", "DB_top100_090", "DB_top100_082"]
CROP_EXTRA = 384
ZOOM_PAD_EXTRA = 30

def compute_doublet_setup(bid):
    """Re-do NB 2 setup for one doublet — returns everything needed to plot a row."""
    row = bench[bench.benchmark_id == bid].iloc[0]
    cx_, cy_ = float(row.x_um), float(row.y_um); cps_focal_ = int(row.cps_id_at_bookmark)
    lin_a_, lin_b_ = row.lineage_pair.split(" × ")
    with tifffile.TiffFile(lib.DAPI_TIF) as tf: H_full, W_full = tf.series[0].shape[-2:]
    yy0, yy1, xx0, xx1 = lib.make_roi_bbox_centred(cx_, cy_, CROP_EXTRA, H_full, W_full)
    dapi_, s18_ = lib.load_morphology(yy0, yy1, xx0, xx1)
    Hl, Wl = s18_.shape
    gene_lin_local = lib.load_lineage_label_map()  # cached internally; cheap
    py_, px_, li_ = lib.load_anchors_in_bbox(yy0, yy1, xx0, xx1, gene_lin_local)
    pi_abst_, conf_, N_eff_ = lib.lineage_posterior_anisotropic(
        py_, px_, li_, Hl, Wl, s18_,
        alpha=lib.ALPHA, n_min=lib.N_MIN,
        n_iter=lib.N_ITER_ANISO, dt=lib.DT_ANISO,
    )
    top_idx_ = np.argmax(pi_abst_, axis=0).astype(np.int8)
    # Colored 18S — blended
    LIN_ARR = np.array(lib.V0_LINEAGE_COLORS[:lib.K], dtype=np.float32)
    _lo, _hi = np.percentile(s18_, [5, 95])
    s18n_ = np.clip((s18_.astype(np.float32) - _lo) / max(_hi - _lo, 1e-6), 0, 1)
    N_tot_ = N_eff_.sum(0)
    pi_ml_ = N_eff_ / np.maximum(N_tot_, 1e-9)[None]
    color_ = np.einsum("khw,kc->hwc", pi_ml_, LIN_ARR)
    c3_ = conf_[..., None].astype(np.float32)
    colored_ = color_ * s18n_[..., None] * c3_ + np.stack([s18n_]*3, axis=-1) * (1 - c3_)
    colored_ = colored_.astype(np.float32)
    # Focal mask + zoom
    wsi_mask_full = tifffile.imread(lib.DATA / "cpsam_whole_slide" / "masks.tif")
    focal_ = (wsi_mask_full[yy0:yy1, xx0:xx1] == cps_focal_); del wsi_mask_full
    if focal_.any():
        ys_, xs_ = np.where(focal_)
        z0 = max(int(ys_.min()) - ZOOM_PAD_EXTRA, 0); z1 = min(int(ys_.max()) + ZOOM_PAD_EXTRA, Hl)
        x0z = max(int(xs_.min()) - ZOOM_PAD_EXTRA, 0); x1z = min(int(xs_.max()) + ZOOM_PAD_EXTRA, Wl)
    else:
        z0, z1, x0z, x1z = 0, Hl, 0, Wl
    zoom_ = (slice(z0, z1), slice(x0z, x1z))
    # Boundary candidates
    b_raw    = find_boundaries(top_idx_, mode="thick").astype(bool)
    b_conf   = b_raw & (conf_ > 0.5)
    # Tanh-gradient (Method 7)
    edge_total_local, _, _ = lib.edge_global_field(pi_abst_)
    evidence_ = lib.cell_evidence(dapi_, s18_, lib.load_wsi_percentiles())
    weighted_e = edge_total_local * (conf_ ** 2) * evidence_
    b_tanh = weighted_e > float(np.percentile(weighted_e, 99))
    return dict(bid=bid, lineage_pair=row.lineage_pair, lin_a=lin_a_, lin_b=lin_b_,
                cps_focal=cps_focal_, colored=colored_, focal=focal_, zoom=zoom_,
                py=py_, px=px_, li=li_, x0z=x0z, z0=z0, x1z=x1z, z1=z1,
                b_raw=b_raw, b_conf=b_conf, b_tanh=b_tanh)

print("computing posteriors and boundaries for", len(EXTRA_DOUBLETS), "doublets…")
t0 = time.time()
doublet_results = []
for bid in EXTRA_DOUBLETS:
    t = time.time()
    try:
        doublet_results.append(compute_doublet_setup(bid))
        print(f"  {bid}: done ({time.time()-t:.0f}s)")
    except Exception as e:
        print(f"  {bid}: FAILED — {e}")
print(f"total: {time.time()-t0:.0f}s")


In [ ]:
COLS = ["colored 18S + anchors (INPUT)", "argmax raw (Method 1)",
        "argmax & conf>0.5 (Method 2)", "tanh-gradient @p99 (Method 7)"]
B_KEYS = [None, "b_raw", "b_conf", "b_tanh"]

nrows = len(doublet_results); ncols = len(COLS)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 4.0 * nrows))
if nrows == 1: axes = axes[None, :]
for r, d in enumerate(doublet_results):
    zoom_d = d["zoom"]
    foc_local = d["focal"][zoom_d]
    foc_bd = find_boundaries(foc_local, mode="outer").astype(int) if foc_local.any() else None
    in_z_d = ((d["py"] >= d["z0"]) & (d["py"] < d["z1"]) &
              (d["px"] >= d["x0z"]) & (d["px"] < d["x1z"]))
    py_z = d["py"][in_z_d] - d["z0"]
    px_z = d["px"][in_z_d] - d["x0z"]
    li_z = d["li"][in_z_d]
    for c, (col_title, key) in enumerate(zip(COLS, B_KEYS)):
        ax = axes[r, c]
        ax.imshow(d["colored"][zoom_d])
        for k_ in range(lib.K):
            m_ = li_z == k_
            if m_.any():
                ax.scatter(px_z[m_], py_z[m_], s=18, color=lib.V0_LINEAGE_COLORS[k_],
                           edgecolors="black", linewidths=0.3, alpha=0.85)
        if key is not None:
            b = d[key]
            ov = np.zeros((zoom_d[0].stop - zoom_d[0].start, zoom_d[1].stop - zoom_d[1].start, 4),
                          dtype=np.float32)
            ov[b[zoom_d]] = (0.0, 1.0, 1.0, 0.65)
            ax.imshow(ov, interpolation="none")
        if foc_bd is not None:
            ax.contour(foc_bd, levels=[0.5], colors="yellow", linewidths=1.3)
        if c == 0:
            ax.set_ylabel(f"{d['bid']}\n{d['lineage_pair']}", fontsize=10)
        if r == 0:
            ax.set_title(col_title, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Multi-doublet boundary validation — zoom on focal, with anchors overlaid on every panel\n"
             "Catches: boundary far from dense-anchor regions (failure mode); boundary correctly between two anchor-color clusters (good).",
             fontsize=12, y=1.001)
plt.tight_layout(); plt.show()


## Validation checklist for boundary estimation

Mark each before moving to notebook 3:

- [ ] The chosen boundary detector lands **inside dense-anchor regions**, not in the empty stroma.
- [ ] Inside cell 018 (focal), the boundary visibly separates the Fib-dominant area from the Mel-dominant area.
- [ ] Off-target boundary count is reasonable (~ few thousand pixels for a 2048 ROI, not 200k+).
- [ ] The boundary is contiguous along real lineage transitions, not fragmented into isolated dots.
- [ ] Threshold knob (τ_c, τ_m, …) chosen with a defensible reason — not just by visual taste.

**Output for notebook 3:** pick one detector and stash its `boundary` mask. Notebook 3 will start from it.